# Clustering 
This code is the full experiment process I went through to decide which clustering method would work the best for my dataset. 

It contains K-means, experimenting K-means with different K values, HDBSCAN and UMAP + HDBSCAN.

In [ ]:
from huggingface_hub import login
from datasets import load_dataset
from datasets import Dataset
from huggingface_hub import login, HfApi
import pandas as pd


import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

classified_dataset = "businessrules/classified_rules"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(classified_dataset)
df_syn = dataset["train"].to_pandas()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/335 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/966 [00:00<?, ? examples/s]

In [ ]:
def get_ids_by_has_rule(x):
    matching_ids = []
    for entry in dataset["train"]:
        if entry['label'] == x:
            matching_ids.append(entry['id'])
    return matching_ids
has_rule_ids = get_ids_by_has_rule(0)

In [ ]:
synthetic_dataset = "businessrules/final_dataset_review"
from datasets import load_dataset

dataset_synth = load_dataset(synthetic_dataset)
syn = dataset_synth["train"].to_pandas()

# K-Means 

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import pandas as pd

def cluster_group0_hasrule(
    df: pd.DataFrame,
    rule_ids: list,
    code_col: str = "cd",
    id_col: str = "id",
    num_clusters: int = 25,
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
):
    """
    Loads a HF dataset, filters group0/HAS_RULE, embeds code, clusters into functional groups,
    and returns a DataFrame containing (id, cluster).
    """


    initial_dataset_size = len(df)
    filtered_rows = []
    for index, row in df.iterrows():
      current_id = row[id_col]
      if current_id in rule_ids:
        filtered_rows.append(row)
    filtered_df = pd.DataFrame(filtered_rows)
    print(f" Initial dataset size: {initial_dataset_size}")
    print(f" Filtered dataset size (after rule_ids filter): {len(filtered_df)}")

    # ---------------------------------------------------------
    # 1. Build embeddings
    # ---------------------------------------------------------
    print(" Loading embedding model:", embedding_model)
    model = SentenceTransformer(embedding_model)

    print(" Embedding code snippets...")
    embeddings = model.encode(syn[code_col].tolist(), batch_size=32, show_progress_bar=True)

    # ---------------------------------------------------------
    # 2. K-Means clustering
    # ---------------------------------------------------------
    print(f" Clustering into {num_clusters} clusters...")
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    clusters = kmeans.fit_predict(embeddings)

    syn["cluster_id"] = clusters

    print(" Clustering completed.")
    print(syn["cluster_id"].value_counts())

    # Return only id and cluster
    cluster_map = filtered_df[[id_col, "cluster_id"]].reset_index(drop=True)
    return cluster_map, embeddings, clusters


In [ ]:
cluster_result, embeddings, clusters = cluster_group0_hasrule( df=syn, rule_ids=has_rule_ids, code_col="cd", id_col="id", num_clusters=5 )

In [ ]:
from sklearn.metrics import silhouette_score
score = silhouette_score(embeddings, clusters)
print("Silhouette Score:", score)

Silhouette Score: 0.047662362


In [ ]:
cluster_result2, embeddings2, clusters2 = cluster_group0_hasrule( df=syn, rule_ids=has_rule_ids, code_col="cd", id_col="id", num_clusters=5, embedding_model = "microsoft/codebert-base")

In [ ]:
from sklearn.metrics import silhouette_score
score2 = silhouette_score(embeddings2, clusters2)
print("Silhouette Score:", score2)

Silhouette Score: 0.12427439


# Experiment different models and Ks

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

def experiment_clustering(
    df: pd.DataFrame,
    rule_ids: list,
    code_col="cd",
    id_col="id",
    k_values=[5, 10, 15, 20, 30],
    embedding_models=[
        "sentence-transformers/all-MiniLM-L6-v2",
        "Salesforce/codet5-base",
        "microsoft/codebert-base"
    ],
    result_csv="/content/drive/MyDrive/cluster_scores.csv"
):

    # Filter rows matching rule_ids
    filtered_df = df[df[id_col].isin(rule_ids)].reset_index(drop=True)
    print(f"Filtered samples: {len(filtered_df)}")

    if filtered_df.empty:
        raise ValueError("No data after filtering rule_ids")

    results = []  # Store experiment rows here

    for model_name in embedding_models:
        print(f"\n Loading model: {model_name}")
        model = SentenceTransformer(model_name)

        print("  Embedding data...")
        embeddings = model.encode(
            filtered_df[code_col].tolist(),
            batch_size=32,
            show_progress_bar=True
        )

        for k in k_values:
            print(f"   Clustering K={k}")
            kmeans = KMeans(n_clusters=k, random_state=42)
            clusters = kmeans.fit_predict(embeddings)

            score = silhouette_score(embeddings, clusters)
            print(f"     Silhouette Score: {score:.4f}")

            # Save to results list
            results.append({
                "model": model_name,
                "k": k,
                "silhouette": score
            })

    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    results_df.to_csv(result_csv, index=False)
    print(f"\n Results saved to Google Drive: {result_csv}")

    return results_df


In [ ]:
results_df = experiment_clustering(
    df=syn,
    rule_ids=has_rule_ids,
    k_values=[5, 10, 15, 20, 30],
    result_csv="/content/drive/MyDrive/cluster_scores.csv"
)


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import os

def experiment_clustering_safe(
    df: pd.DataFrame,
    rule_ids: list,
    code_col="cd",
    id_col="id",
    k_values=[5, 10, 15, 20, 30],
    embedding_models=[
        "sentence-transformers/all-MiniLM-L6-v2",
        "Salesforce/codet5-base",
        "microsoft/codebert-base"
    ],
    drive_csv_path="/content/drive/MyDrive/cluster_scores.csv"
):

    # Filter relevant rows
    filtered_df = df[df[id_col].isin(rule_ids)].reset_index(drop=True)
    if filtered_df.empty:
        raise ValueError("No data after filtering rule_ids")

    # Initialize CSV if it doesn't exist
    if not os.path.exists(drive_csv_path):
        pd.DataFrame(columns=["model", "k", "silhouette"]).to_csv(drive_csv_path, index=False)

    for model_name in embedding_models:
        print(f"\n🔹 Loading model: {model_name}")
        model = SentenceTransformer(model_name)

        print(" → Embedding data...")
        embeddings = model.encode(
            filtered_df[code_col].tolist(),
            batch_size=32,
            show_progress_bar=True
        )

        for k in k_values:
            print(f"   → Clustering K={k}")
            kmeans = KMeans(n_clusters=k, random_state=42)
            clusters = kmeans.fit_predict(embeddings)

            score = silhouette_score(embeddings, clusters)
            print(f"     Silhouette Score: {score:.4f}")

            # Append result immediately
            partial_row = pd.DataFrame([{
                "model": model_name,
                "k": k,
                "silhouette": score
            }])

            partial_row.to_csv(
                drive_csv_path,
                mode="a",        # append mode
                header=False,    # no header (already written)
                index=False
            )

            print(f"     ✔ Progress saved to: {drive_csv_path}")

    print("\n All done — and all progress was saved safely.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:


results_path = "/content/drive/MyDrive/cluster_scores.csv"

experiment_clustering_safe(
    df=syn,
    rule_ids=has_rule_ids,
    k_values=[5, 10, 15, 20, 30],
    drive_csv_path=results_path
)


# Cluster with HDBSCAN

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score
import hdbscan

def cluster_hdbscan_codebert(
    df: pd.DataFrame,
    rule_ids: list,
    code_col="cd",
    id_col="id",
):
    # Filter data
    filtered_df = df[df[id_col].isin(rule_ids)].reset_index(drop=True)
    print(f"Filtered samples: {len(filtered_df)}")

    if filtered_df.empty:
        raise ValueError("No data after filtering rule_ids")

    # Load CodeBERT
    model_name = "microsoft/codebert-base"
    print(f" Loading embedding model: {model_name}")
    model = SentenceTransformer(model_name)

    # Embed code using CodeBERT
    print(" Embedding code...")
    embeddings = model.encode(
        filtered_df[code_col].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    # Run HDBSCAN
    print(" Running HDBSCAN...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=3,
        metric='euclidean'
    )

    clusters = clusterer.fit_predict(embeddings)

    # Add cluster labels
    filtered_df["cluster_id"] = clusters

    # Compute silhouette score (ignore noise cluster -1)
    mask = clusters != -1  # only real clusters
    if mask.sum() > 1:
        score = silhouette_score(embeddings[mask], clusters[mask], metric='cosine')
    else:
        score = None

    print(f"\n Silhouette (cosine, ignoring noise): {score}")

    print("\n Cluster distribution:")
    print(filtered_df["cluster_id"].value_counts())

    return filtered_df, embeddings, clusters, score


In [ ]:
clustered_df, embeddings, cluster_ids, score = cluster_hdbscan_codebert(
    df=syn,
    rule_ids=has_rule_ids,
    code_col="cd",
    id_col="id",
)

Filtered samples: 910
🔹 Loading embedding model: microsoft/codebert-base
🧠 Embedding code...


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

📊 Running HDBSCAN...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



🔍 Silhouette (cosine, ignoring noise): 0.3031255304813385

📊 Cluster distribution:
cluster_id
 1    799
-1    101
 0      5
 2      5
Name: count, dtype: int64


In [ ]:
clustered_df["cluster_id"].value_counts()

,count
cluster_id,
1,799
-1,101
0,5
2,5


# Clustering HDBSCAN + UMAP

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score
import hdbscan
import umap.umap_ as umap  # Import UMAP

def cluster_hdbscan_codebert_umap(
    df: pd.DataFrame,
    rule_ids: list,
    code_col="cd",
    id_col="id",
    # Tunable UMAP parameters
    n_components=10,   # Target dimensions (try 5, 10, or 15)
    n_neighbors=15,    # Size of local neighborhood (lower = more local focus)
    min_dist=0.0       # Minimum distance between points in low-dim space
):
    # 1. Filter data
    filtered_df = df[df[id_col].isin(rule_ids)].reset_index(drop=True)
    print(f"📌 Filtered samples: {len(filtered_df)}")

    if filtered_df.empty:
        print("Warning: No data found after filtering.")
        return pd.DataFrame(), None, None, None

    # 2. Load CodeBERT
    model_name = "microsoft/codebert-base"
    print(f"🔹 Loading embedding model: {model_name}")
    model = SentenceTransformer(model_name)

    # 3. Embed code using CodeBERT (Result: 768 dimensions)
    print("🧠 Generating raw embeddings (768-d)...")
    raw_embeddings = model.encode(
        filtered_df[code_col].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    # 4. Dimensionality Reduction with UMAP (CRITICAL STEP)
    #    This projects the 768-dim fog into a clear lower-dimensional map
    print(f"📉 Applying UMAP to reduce dimensions to {n_components}...")
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=min_dist,
        metric='cosine',  # Use cosine for semantic vectors
        random_state=42   # Fixed seed for reproducibility
    )
    umap_embeddings = reducer.fit_transform(raw_embeddings)

    # 5. Run HDBSCAN on the UMAP embeddings
    print("📊 Running HDBSCAN on UMAP embeddings...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=5,  # Smallest size to consider a cluster
        min_samples=3,       # Measure of how conservative the clustering is
        metric='euclidean',  # UMAP output is Euclidean
        cluster_selection_method='eom' # 'eom' usually finds better natural clusters than 'leaf'
    )

    clusters = clusterer.fit_predict(umap_embeddings)

    # Add cluster labels to dataframe
    filtered_df["cluster_id"] = clusters

    # 6. Compute silhouette score
    # We calculate score on the UMAP embeddings to see how well separated
    # the manifolds are.
    mask = clusters != -1
    if mask.sum() > 1:
        score = silhouette_score(umap_embeddings[mask], clusters[mask], metric='euclidean')
    else:
        score = -1

    print(f"\n🔍 Silhouette Score (on UMAP features): {score:.4f}")

    print("\n📊 Cluster distribution:")
    print(filtered_df["cluster_id"].value_counts().sort_index())

    # Return umap_embeddings so you can plot them later if needed
    return filtered_df, umap_embeddings, clusters, score

In [ ]:
clustered_umap, embeddings_umap, cluster_ids_umap, score_umap = cluster_hdbscan_codebert_umap(
    df=syn,
    rule_ids=has_rule_ids,
    code_col="cd",
    id_col="id",
)

📌 Filtered samples: 910
🔹 Loading embedding model: microsoft/codebert-base
🧠 Generating raw embeddings (768-d)...


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

📉 Applying UMAP to reduce dimensions to 10...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


📊 Running HDBSCAN on UMAP embeddings...

🔍 Silhouette Score (on UMAP features): 0.5925

📊 Cluster distribution:
cluster_id
-1     176
 0      20
 1      29
 2      15
 3       9
      ... 
 57      6
 58     14
 59     12
 60     14
 61     19
Name: count, Length: 63, dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


# HDBSCAN + UMAP less number of clusters

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score
import hdbscan
import umap.umap_ as umap  # Import UMAP

def cluster_hdbscan_codebert_umap_min10(
    df: pd.DataFrame,
    rule_ids: list,
    code_col="cd",
    id_col="id",
    # Tunable UMAP parameters
    n_components=10,   # Target dimensions (try 5, 10, or 15)
    n_neighbors=15,    # Size of local neighborhood (lower = more local focus)
    min_dist=0.0       # Minimum distance between points in low-dim space
):
    # 1. Filter data
    filtered_df = df[df[id_col].isin(rule_ids)].reset_index(drop=True)
    print(f"📌 Filtered samples: {len(filtered_df)}")

    if filtered_df.empty:
        print("Warning: No data found after filtering.")
        return pd.DataFrame(), None, None, None

    # 2. Load CodeBERT
    model_name = "microsoft/codebert-base"
    print(f"🔹 Loading embedding model: {model_name}")
    model = SentenceTransformer(model_name)

    # 3. Embed code using CodeBERT (Result: 768 dimensions)
    print("🧠 Generating raw embeddings (768-d)...")
    raw_embeddings = model.encode(
        filtered_df[code_col].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    # 4. Dimensionality Reduction with UMAP (CRITICAL STEP)
    #    This projects the 768-dim fog into a clear lower-dimensional map
    print(f"📉 Applying UMAP to reduce dimensions to {n_components}...")
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=min_dist,
        metric='cosine',  # Use cosine for semantic vectors
        random_state=42   # Fixed seed for reproducibility
    )
    umap_embeddings = reducer.fit_transform(raw_embeddings)

    # 5. Run HDBSCAN on the UMAP embeddings
    print("📊 Running HDBSCAN on UMAP embeddings...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=10,  # Smallest size to consider a cluster
        min_samples=3,       # Measure of how conservative the clustering is
        metric='euclidean',  # UMAP output is Euclidean
        cluster_selection_method='eom' # 'eom' usually finds better natural clusters than 'leaf'
    )

    clusters = clusterer.fit_predict(umap_embeddings)

    # Add cluster labels to dataframe
    filtered_df["cluster_id"] = clusters

    # 6. Compute silhouette score
    # We calculate score on the UMAP embeddings to see how well separated
    # the manifolds are.
    mask = clusters != -1
    if mask.sum() > 1:
        score = silhouette_score(umap_embeddings[mask], clusters[mask], metric='euclidean')
    else:
        score = -1

    print(f"\n🔍 Silhouette Score (on UMAP features): {score:.4f}")

    print("\n📊 Cluster distribution:")
    print(filtered_df["cluster_id"].value_counts().sort_index())

    # Return umap_embeddings so you can plot them later if needed
    return filtered_df, umap_embeddings, clusters, score

In [ ]:
clustered_umap2, embeddings_umap2, cluster_ids_umap2, score_umap2 = cluster_hdbscan_codebert_umap_min10(
    df=syn,
    rule_ids=has_rule_ids,
    code_col="cd",
    id_col="id",
)

📌 Filtered samples: 910
🔹 Loading embedding model: microsoft/codebert-base
🧠 Generating raw embeddings (768-d)...


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

📉 Applying UMAP to reduce dimensions to 10...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


📊 Running HDBSCAN on UMAP embeddings...

🔍 Silhouette Score (on UMAP features): 0.5699

📊 Cluster distribution:
cluster_id
-1     161
 0      20
 1      29
 2      59
 3      15
 4      12
 5      11
 6      38
 7      19
 8      17
 9      30
 10     17
 11     12
 12     19
 13     33
 14     32
 15     16
 16     43
 17     22
 18     11
 19     10
 20     10
 21     28
 22     28
 23     12
 24     12
 25     21
 26     15
 27     40
 28     22
 29     10
 30     20
 31     45
 32     21
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


# Based on module

In [ ]:
def get_module_ids_by_rule(synthetic_df, rule_ids):
    module_to_ids = {}


    for index, row in syn.iterrows():
        current_id = row['id'] 
        if current_id in rule_ids:
            module_name = row['Module'] 
            if module_name not in module_to_ids:
                module_to_ids[module_name] = []
            module_to_ids[module_name].append(current_id)
    return module_to_ids

module_ids_with_rules = get_module_ids_by_rule(syn, has_rule_ids)
print(module_ids_with_rules)

{'Property': [1, 2, 51, 52, 63, 73, 76, 78, 87, 96, 97, 104, 114, 115, 116, 123, 131, 148, 154, 161, 164, 167, 171, 172, 188, 192, 195, 196, 210, 223, 226, 229, 242, 246, 249, 257, 271, 275, 297, 299, 304, 305, 315, 320, 321, 323, 324, 325, 328, 329, 343, 347, 351, 366, 370, 371, 372, 376, 384, 393, 399, 400, 403, 406, 408, 415, 421, 424, 425, 429, 436, 440, 448, 456, 457, 458, 459, 461, 465, 471, 474, 475, 480, 484, 486, 492, 495, 497, 503, 513, 521, 523, 534, 555, 569, 573, 581, 587, 595, 604, 606, 607, 608, 609, 610, 611, 619, 622, 623, 624, 631, 633, 635, 645, 646, 656, 658, 659, 670, 672, 683, 688, 690, 697, 698, 700, 705, 708, 709, 722, 730, 734, 753, 757, 758, 759, 761, 773, 775, 791, 794, 802, 803, 804, 807, 812, 817, 820, 829, 833, 835, 838, 839, 842, 848, 849, 850, 862, 870, 871, 874, 876, 880, 883, 884, 886, 887, 890, 896, 906, 921, 922, 923, 938, 942, 955, 962, 963, 965, 976, 986, 993, 995, 997], 'MPR': [3, 18, 20, 22, 60, 67, 68, 72, 83, 92, 95, 106, 117, 120, 126, 127, 12